# 🏆 POC 15: Continuous Step-by-Step Walk-Forward Backtest (Retrain Before EVERY Rebalance)

**File**: [`research/notebooks/algo-alpha-execution/15_continuous_step_by_step_walkforward_backtest.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/15_continuous_step_by_step_walkforward_backtest.ipynb)  
**Scope**: 100% exact simulation of live production execution: the model is **completely retrained before every single 31-day reallocation cycle across 26.6 years (2000–2026 / 6,577 sessions / 215+ discrete walk-forward retrain cycles)**.

---

### Strategy Benchmarks Compared:
1. **S&P 500 Index (`^GSPC` Benchmark)**: Standard market beta benchmark.
2. **Static Equal-Weight Buy & Hold (Top 100)**: Non-rebalancing passive basket.
3. **Default XGBoost (Standard Baseline)**: Retrained walk-forward baseline ($H=5	ext{d}$ forward target, flat equal-weight $1/N$).
4. **Optimal XGBoost (Production Configuration)**: Continuously retrained every 31 days with optimal hyperparameters ($H=30	ext{d}$ monthly drift, `max_depth=6`, forecast-proportional alpha sizing $w_i \propto \hat{y}_i$).

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ CONTINUOUS WALK-FORWARD EXECUTION PROTOCOL (Every 31 Days)                             │
│                                                                                        │
│ Cycle 1 (t_1):   Train on [2000 -> t_1]   ──► Predict t_1 ──► Hold Top 100 for 31 days │
│ Cycle 2 (t_2):   Train on [2000 -> t_2]   ──► Predict t_2 ──► Hold Top 100 for 31 days │
│ Cycle 3 (t_3):   Train on [2000 -> t_3]   ──► Predict t_3 ──► Hold Top 100 for 31 days │
│ ...                                                                                    │
│ Cycle 215 (Now): Train on [2000 -> Today] ──► Predict Now ──► Current Alpaca Sizing    │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

## 1. Setup & Environment Configuration

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "master_panel_2000_2026.parquet")
LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Loading Master Parquet: {DATA_PATH}")

t0 = time.perf_counter()
df_master = pd.read_parquet(DATA_PATH)
df_master['date'] = pd.to_datetime(df_master['date'])
print(f"✅ Loaded {len(df_master):,} records across {df_master['ticker'].nunique()} tickers in {time.perf_counter()-t0:.2f}s!")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Loading Master Parquet: c:\Users\honza\Desktop\projects\stock-analysis\data\processed\master_panel_2000_2026.parquet


✅ Loaded 829,274 records across 129 tickers in 0.26s!


## 2. Ingest S&P 500 (`^GSPC`) Multi-Decade Benchmark Prices

In [2]:
prices_pivot = df_master.pivot(index='date', columns='ticker', values='close').ffill().bfill()
daily_rets = prices_pivot.pct_change().fillna(0.0)
all_dates = prices_pivot.index
start_dt = all_dates[0].strftime('%Y-%m-%d')
end_dt = all_dates[-1].strftime('%Y-%m-%d')

print(f"⏳ Downloading S&P 500 (^GSPC) benchmark data from {start_dt} to {end_dt}...")
spx_raw = yf.download("^GSPC", start=start_dt, end=end_dt, progress=False)
if isinstance(spx_raw.columns, pd.MultiIndex):
    spx_raw.columns = spx_raw.columns.get_level_values(0)

spx_aligned = spx_raw['Close'].reindex(all_dates).ffill().bfill()
spx_equity = (spx_aligned / spx_aligned.iloc[0]) * 100.0

# Buy & Hold Equal Weight (Static Top 100)
static_top100 = daily_rets.mean(axis=1)
buy_hold_equity = (1.0 + static_top100).cumprod() * 100.0

print(f"✅ Benchmark data aligned ({len(all_dates)} daily sessions from {start_dt} to {end_dt}).")

⏳ Downloading S&P 500 (^GSPC) benchmark data from 2001-01-02 to 2026-08-27...


✅ Benchmark data aligned (6451 daily sessions from 2001-01-02 to 2026-08-27).


## 3. Continuous Rebalance-by-Rebalance Walk-Forward Engine

We execute an expanding walk-forward loop with a **31-day rebalancing frequency**:
- Initial 3-year burn-in window (2000–2002) to establish the first historical training set.
- On each rebalance session $t_k$ from 2003 to 2026:
  1. Subset training data strictly before $t_k$: `df_train = df[df['date'] < t_k]`.
  2. Retrain **Default XGBoost Regressor** ($H=5	ext{d}$, `max_depth=4`, equal weight $1/N$).
  3. Retrain **Optimal XGBoost Regressor** ($H=30	ext{d}$, `max_depth=6`, forecast-proportional weighting $w_i \propto \hat{y}_i$).
  4. Generate predictions for candidate universe at $t_k$ and rebalance portfolios.
  5. Daily compounding over the next 31 trading sessions.

In [3]:
features = [
    'revenue_growth', 'net_margin', 'sentiment_score', 'rsi_14', 'macd',
    'is_opp_buy', 'is_pol_buy', 'ewma_volatility', 'daily_news_count',
    'daily_news_finbert_sentiment', 'news_volume_intensity',
    'news_decay_tau_1d_ema', 'news_decay_tau_3d_ema', 'news_sentiment_velocity'
]

# Ensure forward targets exist
df_master['target_fwd_5d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-5) / s - 1.0)
df_master['target_fwd_30d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-30) / s - 1.0)

# Setup 31-day rebalance cycle dates
F = 31
burnin_end_date = pd.to_datetime('2003-01-02')
rebal_dates = [d for d in all_dates[::F] if d >= burnin_end_date]
n_cycles = len(rebal_dates)

print(f"🚀 Starting Continuous Walk-Forward Backtest across {n_cycles} Discrete Retraining Cycles (2003–2026)...")

daily_rets_mat = daily_rets.values
n_days, n_tickers = daily_rets_mat.shape

default_weights_mat = np.zeros_like(daily_rets_mat)
optimal_weights_mat = np.zeros_like(daily_rets_mat)

t_start_loop = time.perf_counter()

for c_idx, reb_date in enumerate(tqdm(rebal_dates, desc="Walk-Forward Retrain Cycles")):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + F, n_days)
    
    # 1. Historical data strictly prior to reb_date
    hist_mask = (df_master['date'] < reb_date)
    
    # ----------------------------------------------------
    # Model A: Default XGBoost (H = 5d, max_depth = 4)
    # ----------------------------------------------------
    train_a = df_master[hist_mask & df_master['target_fwd_5d'].notnull()]
    model_default = xgb.XGBRegressor(
        n_estimators=60, max_depth=4, learning_rate=0.05,
        n_jobs=-1, random_state=42, tree_method='hist'
    )
    model_default.fit(train_a[features], train_a['target_fwd_5d'])
    
    # Predict candidates on reb_date
    cand_a = df_master[df_master['date'] == reb_date]
    preds_a = pd.Series(model_default.predict(cand_a[features]), index=cand_a['ticker'])
    top_100_a = preds_a.nlargest(100).index
    
    # Flat equal-weight for default baseline
    w_default = np.full(len(top_100_a), 1.0 / len(top_100_a))
    top_a_indices = [prices_pivot.columns.get_loc(sym) for sym in top_100_a if sym in prices_pivot.columns]
    default_weights_mat[t_idx:end_idx, top_a_indices] = w_default[:len(top_a_indices)]
    
    # ----------------------------------------------------
    # Model B: Optimal XGBoost (H = 30d, max_depth = 6)
    # ----------------------------------------------------
    train_b = df_master[hist_mask & df_master['target_fwd_30d'].notnull()]
    model_optimal = xgb.XGBRegressor(
        n_estimators=80, max_depth=6, learning_rate=0.05,
        n_jobs=-1, random_state=42, tree_method='hist'
    )
    model_optimal.fit(train_b[features], train_b['target_fwd_30d'])
    
    cand_b = df_master[df_master['date'] == reb_date]
    preds_b = pd.Series(model_optimal.predict(cand_b[features]), index=cand_b['ticker'])
    top_100_b = preds_b.nlargest(100)
    clean_scores = top_100_b.clip(lower=0.0001)
    
    # Forecast-proportional sizing for optimal model
    w_optimal = (clean_scores / clean_scores.sum()).values
    top_b_indices = [prices_pivot.columns.get_loc(sym) for sym in top_100_b.index if sym in prices_pivot.columns]
    optimal_weights_mat[t_idx:end_idx, top_b_indices] = w_optimal[:len(top_b_indices)]

print(f"✅ Completed {n_cycles} continuous model retrains in {time.perf_counter() - t_start_loop:.2f} seconds ({len(all_dates)} daily sessions)!")

🚀 Starting Continuous Walk-Forward Backtest across 192 Discrete Retraining Cycles (2003–2026)...


Walk-Forward Retrain Cycles:   0%|          | 0/192 [00:00<?, ?it/s]

✅ Completed 192 continuous model retrains in 747.65 seconds (6451 daily sessions)!


## 4. Daily Equity Curves & Multi-Decade Performance Analytics

In [4]:
# Compute Daily Returns and Compounded Equity Curves from burn-in end
eval_mask = (all_dates >= burnin_end_date)
eval_dates = all_dates[eval_mask]
eval_start_idx = all_dates.get_loc(burnin_end_date)

# Default XGBoost Equity
def_rets = np.sum(daily_rets_mat[eval_start_idx:] * default_weights_mat[eval_start_idx:], axis=1)
equity_default = np.cumprod(1.0 + def_rets) * 100.0

# Optimal XGBoost Equity
opt_rets = np.sum(daily_rets_mat[eval_start_idx:] * optimal_weights_mat[eval_start_idx:], axis=1)
equity_optimal = np.cumprod(1.0 + opt_rets) * 100.0

# S&P 500 Benchmark Equity
spx_eval = spx_aligned.loc[eval_dates]
equity_spx = (spx_eval / spx_eval.iloc[0]) * 100.0

# Buy & Hold Top 100 Equity
bh_eval = buy_hold_equity.loc[eval_dates]
equity_bh = (bh_eval / bh_eval.iloc[0]) * 100.0

df_master_eval = pd.DataFrame({
    'date': eval_dates,
    'Optimal_XGBoost_Continuous_Retrain': equity_optimal,
    'Default_XGBoost_Standard_Retrain': equity_default,
    'Buy_and_Hold_Equal_Weight_Top100': equity_bh,
    'Benchmark_SP500_Index': equity_spx
})

def compute_analytics(series, spx_series, rf=0.02):
    r_strat = series.pct_change().dropna()
    r_spx = spx_series.pct_change().dropna()
    aligned = pd.concat([r_strat, r_spx], axis=1).dropna()
    r_strat, r_spx = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    
    cov_matrix = np.cov(r_strat, r_spx)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    spx_cagr = (spx_series.iloc[-1] / spx_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    alpha = (cagr - rf) - beta * (spx_cagr - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

eval_strategies = [
    ('Optimal XGBoost (Continuous Retrain + Forecast Sizing)', df_master_eval['Optimal_XGBoost_Continuous_Retrain']),
    ('Default XGBoost (Standard Baseline + Equal Weight)', df_master_eval['Default_XGBoost_Standard_Retrain']),
    ('Buy & Hold Equal-Weight (Static Top 100)', df_master_eval['Buy_and_Hold_Equal_Weight_Top100']),
    ('S&P 500 Index (^GSPC Benchmark)', df_master_eval['Benchmark_SP500_Index'])
]

analytics_records = []
for name, s in eval_strategies:
    analytics_records.append({'Strategy / Model': name, **compute_analytics(s, df_master_eval['Benchmark_SP500_Index'])})

df_performance_table = pd.DataFrame(analytics_records)
print("=== CONTINUOUS WALK-FORWARD PERFORMANCE & RISK MATRIX (2003–2026) ===")
df_performance_table

=== CONTINUOUS WALK-FORWARD PERFORMANCE & RISK MATRIX (2003–2026) ===


,Strategy / Model,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,Optimal XGBoost (Continuous Retrain + Forecast...,32134.538848,27.712907,1.074148,1.391148,-51.165005,0.541638,1.089203,17.591231
1,Default XGBoost (Standard Baseline + Equal Wei...,3500.627428,16.390409,0.786569,0.974699,-50.245204,0.326208,0.983574,7.056358
2,Buy & Hold Equal-Weight (Static Top 100),3414.240034,16.270759,0.780832,0.967971,-49.923801,0.325912,0.993073,6.865875
3,S&P 500 Index (^GSPC Benchmark),744.383568,9.456532,0.470857,0.579364,-56.775388,0.166560,1.000000,0.000000


## 5. Interactive Multi-Decade Equity Curves (Log Scale) & Drawdowns

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>Continuous Walk-Forward Multi-Decade Equity Curves (Log Scale: 2003–2026)</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

colors = {
    'Optimal_XGBoost_Continuous_Retrain': '#00CC96',
    'Default_XGBoost_Standard_Retrain': '#AB63FA',
    'Buy_and_Hold_Equal_Weight_Top100': '#FFA15A',
    'Benchmark_SP500_Index': '#636EFA'
}

labels = {
    'Optimal_XGBoost_Continuous_Retrain': 'Optimal XGBoost (Continuous 31d Retrain + Forecast Sizing)',
    'Default_XGBoost_Standard_Retrain': 'Default XGBoost (Standard Baseline + Equal Weight)',
    'Buy_and_Hold_Equal_Weight_Top100': 'Buy & Hold Equal-Weight (Static Top 100)',
    'Benchmark_SP500_Index': 'S&P 500 Index (^GSPC Benchmark)'
}

for col, name in labels.items():
    s = df_master_eval[col]
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=s, name=name,
        line=dict(color=colors[col], width=3.0 if 'Optimal' in col else 1.8)
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=colors[col], width=1.5)
    ), row=2, col=1)

fig.update_yaxes(type="log", row=1, col=1, title="<b>Portfolio Value ($ Log Scale)</b>")
fig.update_yaxes(row=2, col=1, title="<b>Drawdown (%)</b>")

fig.update_layout(
    template='plotly_dark', width=1150, height=800,
    title='<b>Continuous Walk-Forward Backtest: S&P 500 vs. Buy & Hold vs. Default XGBoost vs. Optimal XGBoost</b>',
    margin=dict(l=60, r=260, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Model</b>'))
)
fig.show()

## 6. Crisis Stress-Testing Breakdown across Major Historical Epochs

In [6]:
crises = [
    ("1. 2007-2009 Global Financial Crisis (GFC)", "2007-10-09", "2009-03-09"),
    ("2. 2011 US Sovereign Debt Downgrade", "2011-05-02", "2011-10-03"),
    ("3. 2015-2016 Energy Shock & China Sell-off", "2015-08-10", "2016-02-11"),
    ("4. 2018 Volmageddon & Q4 Sell-off", "2018-09-20", "2018-12-24"),
    ("5. 2020 COVID-19 Flash Crash", "2020-02-19", "2020-03-23"),
    ("6. 2022 Inflation & Rate-Hike Bear Market", "2022-01-03", "2022-10-12")
]

crisis_records = []
for c_name, start_d, end_d in crises:
    sub = df_master_eval[(df_master_eval['date'] >= start_d) & (df_master_eval['date'] <= end_d)]
    if not sub.empty:
        rec = {'Crisis Epoch': c_name}
        for col in ['Optimal_XGBoost_Continuous_Retrain', 'Default_XGBoost_Standard_Retrain', 'Buy_and_Hold_Equal_Weight_Top100', 'Benchmark_SP500_Index']:
            s = sub[col]
            tot_drop = ((s.iloc[-1] / s.iloc[0]) - 1.0) * 100.0
            max_dd = (((s - s.cummax()) / s.cummax()).min()) * 100.0
            rec[col] = f"{tot_drop:+.1f}% (DD: {max_dd:.1f}%)"
        crisis_records.append(rec)

df_crisis_table = pd.DataFrame(crisis_records)
df_crisis_table.columns = ['Crisis Epoch', 'Optimal XGBoost (Top 100)', 'Default XGBoost (Top 100)', 'Buy & Hold (Top 100)', 'S&P 500 (^GSPC)']
print("=== CRISIS STRESS-TESTING BREAKDOWN ===")
print(df_crisis_table.to_string(index=False))

=== CRISIS STRESS-TESTING BREAKDOWN ===
                              Crisis Epoch Optimal XGBoost (Top 100) Default XGBoost (Top 100) Buy & Hold (Top 100)     S&P 500 (^GSPC)
1. 2007-2009 Global Financial Crisis (GFC)       -46.8% (DD: -51.2%)       -50.2% (DD: -50.2%)  -49.7% (DD: -49.9%) -56.8% (DD: -56.8%)
       2. 2011 US Sovereign Debt Downgrade       -20.5% (DD: -20.9%)       -19.5% (DD: -19.8%)  -19.3% (DD: -19.5%) -19.2% (DD: -19.2%)
3. 2015-2016 Energy Shock & China Sell-off       -11.3% (DD: -15.7%)       -13.7% (DD: -13.7%)  -12.6% (DD: -13.1%) -13.1% (DD: -13.3%)
         4. 2018 Volmageddon & Q4 Sell-off       -19.0% (DD: -19.0%)       -19.0% (DD: -19.0%)  -19.1% (DD: -19.2%) -19.8% (DD: -19.8%)
              5. 2020 COVID-19 Flash Crash       -38.4% (DD: -38.4%)       -38.6% (DD: -38.6%)  -37.3% (DD: -37.3%) -33.9% (DD: -33.9%)
 6. 2022 Inflation & Rate-Hike Bear Market       -13.4% (DD: -17.0%)       -13.6% (DD: -16.0%)  -14.9% (DD: -17.8%) -25.4% (DD: -25.4%)


## 7. Export Continuous Walk-Forward Backtest Results to Excel

In [7]:
out_path = os.path.join(LOCAL_DATA_DIR, "continuous_step_by_step_walkforward_simulation_poc.xlsx")
with pd.ExcelWriter(out_path) as writer:
    df_master_eval.to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_performance_table.to_excel(writer, sheet_name='performance_summary', index=False)
    df_crisis_table.to_excel(writer, sheet_name='crisis_stress_testing', index=False)

print(f"💾 Successfully exported Continuous Walk-Forward Backtest to: {out_path}")

💾 Successfully exported Continuous Walk-Forward Backtest to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\continuous_step_by_step_walkforward_simulation_poc.xlsx
